# Praktikum 4
Perbandingan Annoy, FAISS, dan HNSW

Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan membandingkan hasilnya.

In [1]:
import numpy as np
import time
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
np.random.seed(42)
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

In [2]:
# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

=== Annoy ===


Build time: 6.2665698528289795 detik
Query time: 0.0001971721649169922 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...


In [3]:
# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")


=== FAISS (IndexFlatL2) ===
Build time: 0.006346940994262695 detik
Query time: 0.006781101226806641 detik
Neighbors: [901095 495168 561017 163986 185734] ...


In [4]:
# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== HNSW (hnswlib) ===


Build time: 39.74228501319885 detik
Query time: 0.00015115737915039062 detik
Neighbors: [901095 495168 561017 163986 185734] ...


### Percobaan pada Metric Distance Berbeda
Lakukan percobaan pada metric distance yang berbeda. Catat hasilnya pada tabel.

In [5]:
def bench_all(X, query, k, metric):
    n_data, dim = X.shape
    result = {}

    # Annoy
    ann_metric = 'euclidean' if metric == 'l2' else 'angular'
    a = AnnoyIndex(dim, ann_metric)
    start = time.time()
    for i in range(n_data):
        a.add_item(i, X[i])
    a.build(10)
    b_time = time.time() - start
    start = time.time()
    a_idx = a.get_nns_by_vector(query[0], k)
    q_time = time.time() - start
    result['Annoy'] = (a_idx, b_time, q_time)

    # FAISS
    if metric == 'l2':
        f = faiss.IndexFlatL2(dim)
    else:
        f = faiss.IndexFlatIP(dim)
    start = time.time()
    f.add(X)
    b_time = time.time() - start
    start = time.time()
    _, f_idx = f.search(query, k)
    q_time = time.time() - start
    result['FAISS'] = (list(map(int, f_idx[0])), b_time, q_time)

    # HNSW
    space = 'l2' if metric == 'l2' else 'ip'
    h = hnswlib.Index(space=space, dim=dim)
    start = time.time()
    h.init_index(max_elements=n_data, ef_construction=200, M=16)
    h.add_items(X)
    b_time = time.time() - start
    h.set_ef(50)
    start = time.time()
    h_idx, _ = h.knn_query(query, k=k)
    q_time = time.time() - start
    result['HNSW'] = (list(map(int, h_idx[0])), b_time, q_time)

    return result

rows = []
for metric in ['l2', 'inner_product']:
    np.random.seed(42)
    Xs = np.random.random((n_data, dim)).astype(np.float32)
    qs = np.random.random((1, dim)).astype(np.float32)
    res = bench_all(Xs, qs, 10, metric)
    for lib, (idx, b_time, q_time) in res.items():
        rows.append({
            'Metric': metric,
            'Library': lib,
            'Neighbors (top-5)': idx[:5],
            'Build time (s)': round(b_time, 4),
            'Query time (s)': round(q_time, 6),
        })

tabel_cmp = pd.DataFrame(rows)
tabel_cmp

,Metric,Library,Neighbors (top-5),Build time (s),Query time (s)
0,l2,Annoy,"[901095, 495168, 561017, 163986, 185734]",5.9087,0.000083
1,l2,FAISS,"[901095, 495168, 561017, 163986, 185734]",0.0040,0.001977
2,l2,HNSW,"[901095, 495168, 561017, 163986, 185734]",31.6202,0.000076
3,inner_product,Annoy,"[653774, 135588, 561017, 901095, 211473]",7.7653,0.000109
4,inner_product,FAISS,"[958577, 238618, 509273, 52984, 409344]",0.0032,0.001489
5,inner_product,HNSW,"[958577, 238618, 509273, 52984, 409344]",26.7979,0.000074


> **Interpretasi:** FAISS `IndexFlatL2` bersifat *exact* (brute force teroptimasi) sehingga query-nya paling akurat tetapi query time relatif besar. Annoy dan HNSW bersifat *approximate*: build time lebih lama (membuat struktur pohon/graf), tetapi query jauh lebih cepat.